# 001: Linguistic Foundations — Language as probability, n-gram models, and perplexity

## 1. LANGUAGE AS PROBABILITY
- A language model assigns a probability to a sequence of words: $P(w_1, w_2, \ldots, w_n)$.
- Using the chain rule: $P(w_1, \ldots, w_n) = \prod_{i=1}^{n} P(w_i | w_1, \ldots, w_{i-1})$.
- **Markov Assumption** (n-gram): Approximate by conditioning only on the last $k$ words:
  $$P(w_i | w_1, \ldots, w_{i-1}) \approx P(w_i | w_{i-k}, \ldots, w_{i-1})$$
- **Perplexity**: Measures how well a model predicts a sequence. Lower = better:
  $$\text{PP}(W) = P(w_1, \ldots, w_N)^{-1/N}$$
---


In [ ]:
import numpy as np
from collections import Counter, defaultdict

class BigramLanguageModel:
    """Simple bigram (2-gram) language model from scratch."""
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()

    def train(self, corpus):
        """Train on a list of sentences (each a list of tokens)."""
        for sentence in corpus:
            tokens = ['<s>'] + sentence + ['</s>']
            for i in range(len(tokens) - 1):
                self.bigram_counts[tokens[i]][tokens[i+1]] += 1
                self.unigram_counts[tokens[i]] += 1

    def probability(self, word, context):
        """P(word | context) with Laplace smoothing."""
        vocab_size = len(self.unigram_counts)
        count_bigram = self.bigram_counts[context][word]
        count_context = self.unigram_counts[context]
        return (count_bigram + 1) / (count_context + vocab_size)

    def perplexity(self, sentence):
        """Compute perplexity of a sentence."""
        tokens = ['<s>'] + sentence + ['</s>']
        log_prob = 0.0
        N = len(tokens) - 1
        for i in range(N):
            p = self.probability(tokens[i+1], tokens[i])
            log_prob += np.log2(p + 1e-10)
        return 2 ** (-log_prob / N)



In [ ]:
corpus = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "log"],
    ["the", "cat", "ate", "the", "fish"],
]
model = BigramLanguageModel()
model.train(corpus)

print("--- Bigram Probabilities ---")
print(f"P(cat | the) = {model.probability('cat', 'the'):.4f}")
print(f"P(dog | the) = {model.probability('dog', 'the'):.4f}")
print(f"P(sat | cat) = {model.probability('sat', 'cat'):.4f}")

print(f"\n--- Perplexity ---")
print(f"'the cat sat' PP = {model.perplexity(['the', 'cat', 'sat']):.2f}")
print(f"'the fish ate' PP = {model.perplexity(['the', 'fish', 'ate']):.2f}")
